# G4: repository collaboration metrics (`04_repository_collaboration_metrics`)

**Question.** For each repository and day: how much reviewing and discussing happened, on what, and by how many people?

This notebook builds the Gold table `github_lakehouse.gold.repository_collaboration_metrics`.

Contract: `docs/gold_contracts.md`, section 12. The way a Gold table is refreshed is in section 6.

| | |
|---|---|
| Reads | Three Silver tables: `silver.issue_comment_events`, `silver.pull_request_review_events` and `silver.pull_request_review_comment_events` |
| Writes | `gold.repository_collaboration_metrics`, its staging copy `gold_staging.repository_collaboration_metrics`, and one row in `audit.gold_pipeline_runs` |
| Grain | One row per repository per UTC day on which it had at least one event in the three tables |
| Key | `repo_id`, `activity_date` |

The notebook has the same eleven steps as `01_repository_delivery_metrics` and `02_repository_issue_flow`. What is new: it reads **three** Silver tables instead of one. Step 4 places them under each other. From there on, the steps work on one list of events.

## Three kinds of events

| Kind | Silver table | What one event is |
|---|---|---|
| Issue comment | `issue_comment_events` | A comment in the conversation of an issue or of a pull request |
| Review | `pull_request_review_events` | A review of a pull request, with a state such as `approved` |
| Inline review comment | `pull_request_review_comment_events` | A comment on a line of code in a pull request |

## The rules of this table

- **One event is one row and is counted once.** In this week every known action in the three tables is `created`, so the action is not used.
- **Issue comments are split by their target:** `issue`, `pull_request` or unknown. Unknown stays unknown. It is never counted as an issue.
- **Reviews are split by their state:** `approved`, `changes_requested`, `commented`, `dismissed`, or another known state. A review without a state is a review, but it is in none of the five state columns.
- **People are counted once across the three tables.** Someone who comments and reviews in the same repository on the same day is one collaborator.
- **An event without an actor** is an event, but not a person. The coverage column says which share of the events has a known actor.
- **The numbers count events.** Two approving reviews can belong to the same pull request.

## The steps

| # | Step | What it does |
|---|---|---|
| 1 | Parameters | Reads the widgets and chooses the tables |
| 2 | Create the table | `CREATE TABLE IF NOT EXISTS` with the columns of the contract |
| 3 | Affected days | Decides which days this run rebuilds |
| 4 | Read Silver | Places the three tables under each other, only the affected days |
| 5 | Build | The counts per repository and day, the name label, the coverage |
| 6 | Stage | Writes the result to the staging copy |
| 7 | Quality gates | Checks the staging copy and the event IDs |
| 8 | Publish | Replaces the affected days in the Gold table, only if the gates passed |
| 9 | Quality numbers | Totals for the audit row |
| 10 | Audit | Writes one audit row. Stops with an error if the run failed |
| 11 | Results | Shows the audit row and the Gold table |

Before the first run, `08_gold_audit_table` must have run once.

## 1. Parameters

The notebook has the three widgets every Gold notebook has.

| Widget | Blank (default) | With a value |
|---|---|---|
| `source_date` | All days are rebuilt and the whole Gold table is replaced | `YYYY-MM-DD`: only the days touched by that source date are rebuilt and replaced |
| `test_schema` | Production tables | `test_<name>`: every table is read from and written to `github_lakehouse.test_<name>` |
| `run_id` | A generated `manual-...` value | The Job passes its run ID |

### 1.1 Imports and fixed values

In [0]:
import re
import uuid
from datetime import datetime, timezone

# When this run started. It goes into the audit row.
run_started_at = datetime.now(timezone.utc)

# Every time in this notebook is UTC.
spark.conf.set("spark.sql.session.timeZone", "UTC")

PIPELINE_NAME = "gold_repository_collaboration_metrics"

### 1.2 `source_date`

Blank means all dates. A value must be a real date in the form `2024-02-03`. Anything else stops the notebook here, before it reads or writes a table.

In [0]:
dbutils.widgets.text("source_date", "")
source_date = dbutils.widgets.get("source_date").strip()

if source_date == "":
    source_date_value = None
    processing_scope = "ALL_DATES"
else:
    # strptime fails for text that is not a real date, for example 2024-02-31.
    try:
        source_date_value = datetime.strptime(source_date, "%Y-%m-%d").date()
    except ValueError:
        raise ValueError(
            f"Invalid source_date '{source_date}'. "
            "Use a real date such as 2024-02-03, or leave it blank."
        )

    # strptime also accepts 2024-2-3. The text must look exactly like 2024-02-03.
    if str(source_date_value) != source_date:
        raise ValueError(
            f"Invalid source_date '{source_date}'. "
            "Use the form YYYY-MM-DD, for example 2024-02-03."
        )

    processing_scope = "SINGLE_DATE"

print("source_date:", source_date_value)
print("processing_scope:", processing_scope)

### 1.3 `test_schema` and the table names

The six table names are set **only** in this cell: three Silver tables, the Gold table, its staging copy and the audit table. Later cells never change them, otherwise a test run could write to production.

A test schema name must start with `test_` and may contain only lower-case letters, digits and underscores.

In [0]:
dbutils.widgets.text("test_schema", "")
test_schema = dbutils.widgets.get("test_schema").strip()

if test_schema == "":
    ISSUE_COMMENT_TABLE = "github_lakehouse.silver.issue_comment_events"
    REVIEW_TABLE = "github_lakehouse.silver.pull_request_review_events"
    REVIEW_COMMENT_TABLE = "github_lakehouse.silver.pull_request_review_comment_events"
    GOLD_TABLE = "github_lakehouse.gold.repository_collaboration_metrics"
    STAGING_TABLE = "github_lakehouse.gold_staging.repository_collaboration_metrics"
    AUDIT_TABLE = "github_lakehouse.audit.gold_pipeline_runs"
else:
    if re.fullmatch("test_[a-z0-9_]+", test_schema) is None:
        raise ValueError(
            f"Invalid test_schema '{test_schema}'. "
            "It must start with test_ and may contain only lower-case letters, digits and underscores."
        )
    ISSUE_COMMENT_TABLE = f"github_lakehouse.{test_schema}.issue_comment_events"
    REVIEW_TABLE = f"github_lakehouse.{test_schema}.pull_request_review_events"
    REVIEW_COMMENT_TABLE = f"github_lakehouse.{test_schema}.pull_request_review_comment_events"
    GOLD_TABLE = f"github_lakehouse.{test_schema}.repository_collaboration_metrics"
    STAGING_TABLE = f"github_lakehouse.{test_schema}.repository_collaboration_metrics_staging"
    AUDIT_TABLE = f"github_lakehouse.{test_schema}.gold_pipeline_runs"

print("Issue comment table :", ISSUE_COMMENT_TABLE)
print("Review table        :", REVIEW_TABLE)
print("Review comment table:", REVIEW_COMMENT_TABLE)
print("Gold table          :", GOLD_TABLE)
print("Staging table       :", STAGING_TABLE)
print("Audit table         :", AUDIT_TABLE)

### 1.4 `run_id`

The Job passes its run ID. A run by hand gets a generated ID. The ID is written into every Gold row, so it may contain only letters, digits, `-` and `_`.

In [0]:
dbutils.widgets.text("run_id", "")
run_id = dbutils.widgets.get("run_id").strip()

if run_id == "":
    run_id = "manual-" + str(uuid.uuid4())

if re.fullmatch("[A-Za-z0-9_-]+", run_id) is None:
    raise ValueError(
        f"Invalid run_id '{run_id}'. "
        "It may contain only letters, digits, - and _."
    )

print("run_id:", run_id)

### 1.5 Is the setup done?

The audit table is created by `08_gold_audit_table`. The three Silver tables are created by the Silver notebooks `04`, `05` and `06`. If one of the four tables does not exist, the notebook stops here with a clear message.

In [0]:
if not spark.catalog.tableExists(AUDIT_TABLE):
    raise RuntimeError(
        f"{AUDIT_TABLE} does not exist. Run 08_gold_audit_table first."
    )

if not spark.catalog.tableExists(ISSUE_COMMENT_TABLE):
    raise RuntimeError(
        f"{ISSUE_COMMENT_TABLE} does not exist. Run the Silver notebook 04_issue_comment_events first."
    )

if not spark.catalog.tableExists(REVIEW_TABLE):
    raise RuntimeError(
        f"{REVIEW_TABLE} does not exist. Run the Silver notebook 05_pull_request_review_events first."
    )

if not spark.catalog.tableExists(REVIEW_COMMENT_TABLE):
    raise RuntimeError(
        f"{REVIEW_COMMENT_TABLE} does not exist. Run the Silver notebook 06_pull_request_review_comment_events first."
    )

print("Setup found  :", AUDIT_TABLE)
print("Sources found:", ISSUE_COMMENT_TABLE)
print("              ", REVIEW_TABLE)
print("              ", REVIEW_COMMENT_TABLE)

## 2. Create the Gold table

The columns, their types and their order are those of the contract. `CREATE TABLE IF NOT EXISTS` does nothing when the table already exists, so this step is safe to run again.

| Column | Meaning |
|---|---|
| `repo_id` | Key |
| `activity_date` | Key. The UTC date of the events |
| `repo_name` | Label: the latest name seen that day |
| `issue_comment_events` | All issue comment events that day |
| `issue_conversation_comment_events` | Of those, comments on issues |
| `pr_conversation_comment_events` | Of those, comments in pull request conversations |
| `unclassified_conversation_comment_events` | Of those, comments with an unknown target |
| `review_events` | All review events that day |
| `review_state_known_events` | Reviews with a known state |
| `review_approved_events` | Reviews with the state `approved` |
| `review_changes_requested_events` | Reviews with the state `changes_requested` |
| `review_commented_events` | Reviews with the state `commented` |
| `review_dismissed_events` | Reviews with the state `dismissed` |
| `review_other_state_events` | Reviews with a known state that is none of the four, for example `pending` |
| `inline_review_comment_events` | All inline review comment events that day |
| `collaboration_events_total` | The three kinds of events together |
| `actor_known_events` | Events with a known actor. The base of the two columns below |
| `unique_collaborating_actors` | Different actors across the three kinds of events |
| `collaborator_actor_coverage_pct` | 100 × events with a known actor ÷ all events |
| `_gold_run_id` | The run that last wrote the row |
| `_gold_updated_at` | When that run wrote it |

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {GOLD_TABLE} (
        -- Key and label
        repo_id                                   BIGINT,
        activity_date                             DATE,
        repo_name                                 STRING,

        -- Issue comment events by target
        issue_comment_events                      BIGINT,
        issue_conversation_comment_events         BIGINT,
        pr_conversation_comment_events            BIGINT,
        unclassified_conversation_comment_events  BIGINT,

        -- Review events by state
        review_events                             BIGINT,
        review_state_known_events                 BIGINT,
        review_approved_events                    BIGINT,
        review_changes_requested_events           BIGINT,
        review_commented_events                   BIGINT,
        review_dismissed_events                   BIGINT,
        review_other_state_events                 BIGINT,

        -- Inline review comment events
        inline_review_comment_events              BIGINT,

        -- All three together, and the people
        collaboration_events_total                BIGINT,
        actor_known_events                        BIGINT,
        unique_collaborating_actors               BIGINT,
        collaborator_actor_coverage_pct           DOUBLE,

        -- Technical columns
        _gold_run_id                              STRING,
        _gold_updated_at                          TIMESTAMP
    )
    USING DELTA
""")

print("Table ready:", GOLD_TABLE)

## 3. Affected days

A Gold row is a summary of a whole day. When one new event arrives for a day, the row of that day has to be calculated again from **all** events of that day. So the first question of every run is: which days does this run have to rebuild?

### 3.1 The Silver rows in the scope of this run

- With a `source_date`, the scope is the Silver rows that were loaded from that source date.
- With a blank date, the scope is all Silver rows.

This table reads three Silver tables, so the scope is taken from all three. `UNION ALL` places the results of the three queries under each other. Only the event date is needed here.

A row without an event date cannot be placed on a day. It is left out and counted here, and the count goes into the audit row. The profiling run found 0 such rows.

In [0]:
if source_date_value is None:
    scope_filter = "true"
else:
    scope_filter = f"_source_date = DATE'{source_date}'"

scope_rows = spark.sql(f"""
    SELECT event_date
    FROM {ISSUE_COMMENT_TABLE}
    WHERE {scope_filter}

    UNION ALL

    SELECT event_date
    FROM {REVIEW_TABLE}
    WHERE {scope_filter}

    UNION ALL

    SELECT event_date
    FROM {REVIEW_COMMENT_TABLE}
    WHERE {scope_filter}
""")
scope_rows.createOrReplaceTempView("scope_rows")

scope_row = spark.sql("""
    SELECT
        COUNT(*)                      AS rows_in_scope,
        count_if(event_date IS NULL)  AS rows_without_event_date
    FROM scope_rows
""").collect()[0]

rows_in_scope = scope_row["rows_in_scope"]
rows_without_event_date = scope_row["rows_without_event_date"]

print("Scope filter                :", scope_filter)
print("Silver rows in the scope    :", rows_in_scope)
print("Rows without an event date  :", rows_without_event_date)

### 3.2 The days these rows belong to

The affected days are the event dates of the rows in the scope, from any of the three tables. A day is rebuilt also when only one of the three tables has a new row for it.

In the week that is loaded, the event date and the source date of a row are always the same day (measured by the profiling run). So a run for one source date rebuilds exactly one day.

In [0]:
day_rows = spark.sql("""
    SELECT DISTINCT event_date
    FROM scope_rows
    WHERE event_date IS NOT NULL
    ORDER BY event_date
""").collect()

affected_dates = []
for day_row in day_rows:
    affected_dates.append(day_row["event_date"])

print("Affected days:", len(affected_dates))
for affected_date in affected_dates:
    print("  ", affected_date)

### 3.3 The day list as SQL text

The later steps need the affected days inside a SQL `IN (...)` list, for example `DATE'2024-02-01', DATE'2024-02-02'`. The same text is used to read Silver, to check the result and to publish, so all three work on exactly the same days.

When there is no affected day, the text is `NULL`. `IN (NULL)` is never true, so the later queries simply find no row.

In [0]:
if len(affected_dates) == 0:
    affected_days_sql = "NULL"
else:
    day_texts = []
    for affected_date in affected_dates:
        day_texts.append("DATE'" + str(affected_date) + "'")
    affected_days_sql = ", ".join(day_texts)

print("Day list:", affected_days_sql)

## 4. Read Silver

All Silver rows of the affected days are read, **not only the rows of this source date**. A row that arrived earlier for the same day must be counted again, otherwise the new Gold row would be too small.

The three tables are placed under each other with `UNION ALL`. They are **not joined**: a join of raw events from two tables would multiply the rows.

`UNION ALL` matches the columns by their position, not by their name. So the three queries select the same columns in the same order:

| Column | Issue comments | Reviews | Inline review comments |
|---|---|---|---|
| `event_kind` | `'issue_comment'` | `'review'` | `'review_comment'` |
| `comment_target_type` | from Silver | NULL | NULL |
| `review_state` | NULL | from Silver | NULL |
| the other six columns | from Silver | from Silver | from Silver |

`event_kind` is a fixed text that says which table a row comes from. A column that a table does not have is filled with NULL. `CAST(NULL AS STRING)` gives that NULL the type of the column.

Only the columns this table needs are selected. `event_date` gets its Gold name `activity_date`.

Spark does not read anything yet. A temporary view is only a name for a query. The work is done in step 6, when the result is written.

In [0]:
collaboration_rows = spark.sql(f"""
    SELECT
        'issue_comment'          AS event_kind,
        event_id,
        event_timestamp,
        event_date               AS activity_date,
        repo_id,
        repo_name,
        actor_id,
        comment_target_type,
        CAST(NULL AS STRING)     AS review_state
    FROM {ISSUE_COMMENT_TABLE}
    WHERE event_date IN ({affected_days_sql})

    UNION ALL

    SELECT
        'review'                 AS event_kind,
        event_id,
        event_timestamp,
        event_date               AS activity_date,
        repo_id,
        repo_name,
        actor_id,
        CAST(NULL AS STRING)     AS comment_target_type,
        review_state
    FROM {REVIEW_TABLE}
    WHERE event_date IN ({affected_days_sql})

    UNION ALL

    SELECT
        'review_comment'         AS event_kind,
        event_id,
        event_timestamp,
        event_date               AS activity_date,
        repo_id,
        repo_name,
        actor_id,
        CAST(NULL AS STRING)     AS comment_target_type,
        CAST(NULL AS STRING)     AS review_state
    FROM {REVIEW_COMMENT_TABLE}
    WHERE event_date IN ({affected_days_sql})
""")
collaboration_rows.createOrReplaceTempView("collaboration_rows")

### 4.1 How many Silver rows these days have

The rows are counted directly on each Silver table, not on the view above. So these numbers do not depend on the steps that follow.

They go into the audit row, one number per table. The quality gates compare each of them with the sum of its Gold column: every Silver row of the affected days must be counted exactly once.

In [0]:
issue_comment_row_count = spark.sql(f"""
    SELECT COUNT(*) AS silver_rows
    FROM {ISSUE_COMMENT_TABLE}
    WHERE event_date IN ({affected_days_sql})
""").collect()[0]["silver_rows"]

review_row_count = spark.sql(f"""
    SELECT COUNT(*) AS silver_rows
    FROM {REVIEW_TABLE}
    WHERE event_date IN ({affected_days_sql})
""").collect()[0]["silver_rows"]

review_comment_row_count = spark.sql(f"""
    SELECT COUNT(*) AS silver_rows
    FROM {REVIEW_COMMENT_TABLE}
    WHERE event_date IN ({affected_days_sql})
""").collect()[0]["silver_rows"]

silver_row_count = issue_comment_row_count + review_row_count + review_comment_row_count

print("Issue comment rows on the affected days        :", issue_comment_row_count)
print("Review rows on the affected days               :", review_row_count)
print("Inline review comment rows on the affected days:", review_comment_row_count)
print("All three together                             :", silver_row_count)

## 5. Build

The Gold rows are built in three small queries. Each one is saved as a temporary view, and a later one reads it.

| Query | View | One row per |
|---|---|---|
| 5.1 | `collaboration_counts` | Repository and day: all the counts |
| 5.2 | `repo_names` | Repository and day: the name label |
| 5.3 | `gold_rows` | Repository and day: everything together, with the coverage |

### 5.1 The counts per repository and day

One `GROUP BY` over the stacked events gives one row per repository and day.

How the functions treat NULL:

- `COUNT(*)` counts every row.
- `count_if(condition)` counts the rows where the condition is true. A comparison with NULL is never true. So a review without a state is counted in `review_events`, but in none of the state columns.
- `COUNT(actor_id)` counts only the rows where the actor is known.
- `COUNT(DISTINCT actor_id)` counts the different actors and leaves NULL out. With no known actor it gives 0.

**Issue comments by target.** The three target columns are `issue`, `pull_request` and unknown (NULL). Silver writes only these three. If another value ever appears, the three columns no longer add up, and the quality gate `TARGETS_DO_NOT_ADD_UP` stops the run.

**Reviews by state.** `NOT IN ('approved', 'changes_requested', 'commented', 'dismissed')` finds a known state that is none of the four, for example `pending`. A state is compared exactly as Silver stores it, in lower-case letters.

**People.** `COUNT(DISTINCT actor_id)` runs over all three kinds of events at once. That is why the tables were stacked first: someone who comments and reviews on the same day is counted once.

In [0]:
collaboration_counts = spark.sql("""
    SELECT
        repo_id,
        activity_date,

        count_if(event_kind = 'issue_comment')
            AS issue_comment_events,
        count_if(event_kind = 'issue_comment' AND comment_target_type = 'issue')
            AS issue_conversation_comment_events,
        count_if(event_kind = 'issue_comment' AND comment_target_type = 'pull_request')
            AS pr_conversation_comment_events,
        count_if(event_kind = 'issue_comment' AND comment_target_type IS NULL)
            AS unclassified_conversation_comment_events,

        count_if(event_kind = 'review')
            AS review_events,
        count_if(event_kind = 'review' AND review_state IS NOT NULL)
            AS review_state_known_events,
        count_if(event_kind = 'review' AND review_state = 'approved')
            AS review_approved_events,
        count_if(event_kind = 'review' AND review_state = 'changes_requested')
            AS review_changes_requested_events,
        count_if(event_kind = 'review' AND review_state = 'commented')
            AS review_commented_events,
        count_if(event_kind = 'review' AND review_state = 'dismissed')
            AS review_dismissed_events,
        count_if(event_kind = 'review'
                 AND review_state NOT IN ('approved', 'changes_requested', 'commented', 'dismissed'))
            AS review_other_state_events,

        count_if(event_kind = 'review_comment')
            AS inline_review_comment_events,

        COUNT(*)                  AS collaboration_events_total,
        COUNT(actor_id)           AS actor_known_events,
        COUNT(DISTINCT actor_id)  AS unique_collaborating_actors

    FROM collaboration_rows
    GROUP BY repo_id, activity_date
""")
collaboration_counts.createOrReplaceTempView("collaboration_counts")

### 5.2 The `repo_name` label

A repository can be renamed, so `repo_name` is only a label and never a key. The label of a day is the name on the **latest** event of that repository on that day, whichever of the three tables it comes from.

`ROW_NUMBER` numbers the events of each repository and day, the latest first. Number 1 is the one we keep.

- Events without a name are left out first, so a missing name is skipped.
- If two events have the same time, the one with the larger `event_id` gets number 1. `event_id` is text and is compared as text. This makes the result the same on every run.

In [0]:
repo_names = spark.sql("""
    SELECT
        repo_id,
        activity_date,
        repo_name
    FROM (
        SELECT
            repo_id,
            activity_date,
            repo_name,
            ROW_NUMBER() OVER (
                PARTITION BY repo_id, activity_date
                ORDER BY event_timestamp DESC, event_id DESC
            ) AS name_rank
        FROM collaboration_rows
        WHERE repo_name IS NOT NULL
    ) ranked_names
    WHERE name_rank = 1
""")
repo_names.createOrReplaceTempView("repo_names")

### 5.3 The Gold rows

The counts and the label are put together. The columns are in the order of the Gold table.

- `LEFT JOIN` keeps a repository-day whose events all have no name. Its label is NULL.
- **Coverage:** 100 × events with a known actor ÷ all events, rounded to 2 decimals. In SQL, `/` gives a decimal number also when both sides are whole numbers. The `CASE` keeps the query from dividing by zero. A row always has at least one event, so the coverage is never NULL here.
- A coverage of 0 means: the day has events, but none of them has a known actor. Then the number of people is 0 too.
- The two technical columns say which run wrote the row and when.

In [0]:
gold_rows = spark.sql(f"""
    SELECT
        c.repo_id,
        c.activity_date,
        n.repo_name,

        c.issue_comment_events,
        c.issue_conversation_comment_events,
        c.pr_conversation_comment_events,
        c.unclassified_conversation_comment_events,

        c.review_events,
        c.review_state_known_events,
        c.review_approved_events,
        c.review_changes_requested_events,
        c.review_commented_events,
        c.review_dismissed_events,
        c.review_other_state_events,

        c.inline_review_comment_events,

        c.collaboration_events_total,
        c.actor_known_events,
        c.unique_collaborating_actors,
        CASE
            WHEN c.collaboration_events_total > 0
            THEN ROUND(100 * c.actor_known_events / c.collaboration_events_total, 2)
        END AS collaborator_actor_coverage_pct,

        '{run_id}'           AS _gold_run_id,
        current_timestamp()  AS _gold_updated_at

    FROM collaboration_counts c
    LEFT JOIN repo_names n
      ON  n.repo_id = c.repo_id
      AND n.activity_date = c.activity_date
""")
gold_rows.createOrReplaceTempView("gold_rows")

## 6. Stage

The result is written to the staging copy first. The Gold table is not touched yet.

Spark does its work only now, when the rows are written. If something goes wrong, the error is kept in `stage_error`, so that the audit row at the end can report it.

In [0]:
stage_error = None

try:
    spark.sql(f"""
        CREATE OR REPLACE TABLE {STAGING_TABLE} AS
        SELECT * FROM gold_rows
    """)
    print("Staged:", STAGING_TABLE)

except Exception as error:
    stage_error = str(error)[:1000]
    print("Staging FAILED:", stage_error)

## 7. Quality gates

The checks run on the staging copy. One of them, the event IDs, runs on the stacked Silver rows. If a gate fails, nothing is published.

| Gate | It fails when |
|---|---|
| `KEY_NOT_UNIQUE` | A repository and day appear more than once |
| `KEY_IS_NULL` | A row has no `repo_id` or no `activity_date` |
| `COUNT_IS_NULL` | One of the 15 count columns is missing. A count is 0 when nothing was counted, never NULL |
| `TARGETS_DO_NOT_ADD_UP` | Issue + pull request + unknown target is not the number of issue comment events |
| `STATES_DO_NOT_ADD_UP` | The five state columns together are not the number of reviews with a known state |
| `KNOWN_STATES_ABOVE_REVIEWS` | A row has more reviews with a known state than reviews |
| `TOTAL_IS_NOT_THE_SUM` | The total is not issue comments + reviews + inline review comments |
| `ACTOR_COUNTS_OUT_OF_ORDER` | Different actors ≤ events with a known actor ≤ all events is not true, or a row has events with a known actor but 0 actors |
| `COVERAGE_OUT_OF_RANGE` | A coverage is below 0 or above 100 |
| `COVERAGE_DOES_NOT_FIT_ITS_BASE` | A row with events has no coverage, or a row without events has one |
| `EVENT_ID_NOT_UNIQUE` | An `event_id` appears more than once in the three Silver tables together, on the affected days |
| `EVENT_SUM_DIFFERS_FROM_SILVER` | The sum of one of the three event columns is not the number of rows of its Silver table on the affected days |

### 7.1 Keys that appear more than once

In [0]:
duplicate_key_count = 0

if stage_error is None:
    duplicate_key_count = spark.sql(f"""
        SELECT COUNT(*) AS duplicate_keys
        FROM (
            SELECT repo_id, activity_date
            FROM {STAGING_TABLE}
            GROUP BY repo_id, activity_date
            HAVING COUNT(*) > 1
        ) keys_seen_twice
    """).collect()[0]["duplicate_keys"]

print("Keys that appear more than once:", duplicate_key_count)

### 7.2 Event IDs that appear more than once

Each Silver table has every `event_id` only once. Here the three tables are stacked, so the same check is made on all three together. If one event were in two tables, it would be counted twice in the total.

This check reads the stacked Silver rows of step 4, not the staging copy, because the Gold rows no longer carry event IDs. The profiling run found no `event_id` in more than one Silver table.

In [0]:
duplicate_event_id_count = 0

if stage_error is None:
    duplicate_event_id_count = spark.sql("""
        SELECT COUNT(*) AS duplicate_event_ids
        FROM (
            SELECT event_id
            FROM collaboration_rows
            GROUP BY event_id
            HAVING COUNT(*) > 1
        ) event_ids_seen_twice
    """).collect()[0]["duplicate_event_ids"]

print("Event IDs that appear more than once:", duplicate_event_id_count)

### 7.3 The other checks

One query counts, for every rule, the rows that break it. The expected value of each count is 0. The last four values are the row count and the sums of the three event columns.

`COALESCE(SUM(...), 0)`: the sum of no rows is NULL. For the comparison with Silver it has to be 0.

The comparison with Silver is made three times, once per table: the sum of `issue_comment_events` with the issue comment rows, the sum of `review_events` with the review rows, and the sum of `inline_review_comment_events` with the inline review comment rows.

In [0]:
failed_checks = []
staged_row_count = 0
staged_issue_comment_sum = 0
staged_review_sum = 0
staged_review_comment_sum = 0

if stage_error is None:
    gate_row = spark.sql(f"""
        SELECT
            count_if(repo_id IS NULL OR activity_date IS NULL)
                AS rows_without_key,

            count_if(issue_comment_events IS NULL
                  OR issue_conversation_comment_events IS NULL
                  OR pr_conversation_comment_events IS NULL
                  OR unclassified_conversation_comment_events IS NULL
                  OR review_events IS NULL
                  OR review_state_known_events IS NULL
                  OR review_approved_events IS NULL
                  OR review_changes_requested_events IS NULL
                  OR review_commented_events IS NULL
                  OR review_dismissed_events IS NULL
                  OR review_other_state_events IS NULL
                  OR inline_review_comment_events IS NULL
                  OR collaboration_events_total IS NULL
                  OR actor_known_events IS NULL
                  OR unique_collaborating_actors IS NULL)
                AS rows_with_missing_count,

            count_if(issue_conversation_comment_events
                     + pr_conversation_comment_events
                     + unclassified_conversation_comment_events
                     <> issue_comment_events)
                AS rows_where_targets_do_not_add_up,

            count_if(review_approved_events
                     + review_changes_requested_events
                     + review_commented_events
                     + review_dismissed_events
                     + review_other_state_events
                     <> review_state_known_events)
                AS rows_where_states_do_not_add_up,

            count_if(review_state_known_events > review_events)
                AS rows_with_known_states_above_reviews,

            count_if(issue_comment_events
                     + review_events
                     + inline_review_comment_events
                     <> collaboration_events_total)
                AS rows_where_total_is_not_the_sum,

            count_if(unique_collaborating_actors > actor_known_events
                  OR actor_known_events > collaboration_events_total
                  OR (actor_known_events > 0 AND unique_collaborating_actors = 0))
                AS rows_with_actor_counts_out_of_order,

            count_if(collaborator_actor_coverage_pct < 0
                  OR collaborator_actor_coverage_pct > 100)
                AS rows_with_coverage_out_of_range,

            count_if((collaboration_events_total > 0 AND collaborator_actor_coverage_pct IS NULL)
                  OR (collaboration_events_total = 0 AND collaborator_actor_coverage_pct IS NOT NULL))
                AS rows_where_coverage_does_not_fit,

            COUNT(*)                                        AS staged_rows,
            COALESCE(SUM(issue_comment_events), 0)          AS staged_issue_comment_sum,
            COALESCE(SUM(review_events), 0)                 AS staged_review_sum,
            COALESCE(SUM(inline_review_comment_events), 0)  AS staged_review_comment_sum

        FROM {STAGING_TABLE}
    """).collect()[0]

    staged_row_count = gate_row["staged_rows"]
    staged_issue_comment_sum = gate_row["staged_issue_comment_sum"]
    staged_review_sum = gate_row["staged_review_sum"]
    staged_review_comment_sum = gate_row["staged_review_comment_sum"]

    if duplicate_key_count > 0:
        failed_checks.append("KEY_NOT_UNIQUE")
    if gate_row["rows_without_key"] > 0:
        failed_checks.append("KEY_IS_NULL")
    if gate_row["rows_with_missing_count"] > 0:
        failed_checks.append("COUNT_IS_NULL")
    if gate_row["rows_where_targets_do_not_add_up"] > 0:
        failed_checks.append("TARGETS_DO_NOT_ADD_UP")
    if gate_row["rows_where_states_do_not_add_up"] > 0:
        failed_checks.append("STATES_DO_NOT_ADD_UP")
    if gate_row["rows_with_known_states_above_reviews"] > 0:
        failed_checks.append("KNOWN_STATES_ABOVE_REVIEWS")
    if gate_row["rows_where_total_is_not_the_sum"] > 0:
        failed_checks.append("TOTAL_IS_NOT_THE_SUM")
    if gate_row["rows_with_actor_counts_out_of_order"] > 0:
        failed_checks.append("ACTOR_COUNTS_OUT_OF_ORDER")
    if gate_row["rows_with_coverage_out_of_range"] > 0:
        failed_checks.append("COVERAGE_OUT_OF_RANGE")
    if gate_row["rows_where_coverage_does_not_fit"] > 0:
        failed_checks.append("COVERAGE_DOES_NOT_FIT_ITS_BASE")
    if duplicate_event_id_count > 0:
        failed_checks.append("EVENT_ID_NOT_UNIQUE")
    if (staged_issue_comment_sum != issue_comment_row_count
            or staged_review_sum != review_row_count
            or staged_review_comment_sum != review_comment_row_count):
        failed_checks.append("EVENT_SUM_DIFFERS_FROM_SILVER")

print("Staged rows (repository-days)       :", staged_row_count)
print("Sum of issue_comment_events         :", staged_issue_comment_sum, "| Silver rows:", issue_comment_row_count)
print("Sum of review_events                :", staged_review_sum, "| Silver rows:", review_row_count)
print("Sum of inline_review_comment_events :", staged_review_comment_sum, "| Silver rows:", review_comment_row_count)

if len(failed_checks) == 0:
    print("Quality gates: no gate failed")
else:
    print("Quality gates FAILED:", "; ".join(failed_checks))

## 8. Publish

What happens with the staged rows:

| Situation | Status | Gold table |
|---|---|---|
| Staging failed | `FAILED` | Unchanged |
| There is no affected day, because Silver has no rows in the scope of this run | `NO_INPUT` | Unchanged |
| A quality gate failed | `FAILED` | Unchanged |
| Everything is fine | `SUCCEEDED` | The affected days are replaced by the staged rows |

How the days are replaced:

- **With a `source_date`:** `INSERT INTO ... REPLACE WHERE activity_date IN (...)` deletes the rows of the affected days and inserts the staged rows. The other days are not touched. Delta also refuses a staged row that belongs to another day.
- **With a blank date:** `INSERT OVERWRITE` replaces the whole table. This also removes a day that no longer exists in Silver.

Both are one Delta commit. A reader sees either the old rows or the new ones, never a half-written day. A repository-day that no longer has events disappears, because its whole day is replaced.

In [0]:
run_status = None
error_message = None
published_rows = 0

if stage_error is not None:
    run_status = "FAILED"
    error_message = "Staging failed: " + stage_error

elif len(affected_dates) == 0:
    run_status = "NO_INPUT"

elif len(failed_checks) > 0:
    run_status = "FAILED"
    error_message = "Quality gates failed: " + "; ".join(failed_checks)

else:
    # The first line of the statement depends on the scope of the run.
    if source_date_value is None:
        publish_start = f"INSERT OVERWRITE TABLE {GOLD_TABLE}"
    else:
        publish_start = (
            f"INSERT INTO {GOLD_TABLE} "
            f"REPLACE WHERE activity_date IN ({affected_days_sql})"
        )

    print("Publish statement starts with:", publish_start)

    try:
        spark.sql(f"""
            {publish_start}
            SELECT
                repo_id,
                activity_date,
                repo_name,
                issue_comment_events,
                issue_conversation_comment_events,
                pr_conversation_comment_events,
                unclassified_conversation_comment_events,
                review_events,
                review_state_known_events,
                review_approved_events,
                review_changes_requested_events,
                review_commented_events,
                review_dismissed_events,
                review_other_state_events,
                inline_review_comment_events,
                collaboration_events_total,
                actor_known_events,
                unique_collaborating_actors,
                collaborator_actor_coverage_pct,
                _gold_run_id,
                _gold_updated_at
            FROM {STAGING_TABLE}
        """)
        run_status = "SUCCEEDED"
        published_rows = staged_row_count

    except Exception as error:
        run_status = "FAILED"
        error_message = "Publish failed: " + str(error)[:1000]

print("Status:", run_status)
print("Published rows:", published_rows)

## 9. Quality numbers for the audit row

These totals describe what this run calculated. They are counted on the staging copy, so they are there also when the run was not published. None of them stops the run.

After a run for all dates they can be compared directly with the expected values in section 12 of the contract.

`unique_collaborating_actors` summed over the rows counts a person once for every repository and day on which they took part. It is a number of person-repository-days, not of people.

In [0]:
quality_metrics = {
    "repository_days": 0,
    "issue_comment_events": 0,
    "issue_conversation_comment_events": 0,
    "pr_conversation_comment_events": 0,
    "unclassified_conversation_comment_events": 0,
    "review_events": 0,
    "review_state_known_events": 0,
    "review_approved_events": 0,
    "review_changes_requested_events": 0,
    "review_commented_events": 0,
    "review_dismissed_events": 0,
    "review_other_state_events": 0,
    "inline_review_comment_events": 0,
    "collaboration_events_total": 0,
    "actor_known_events": 0,
    "unique_collaborating_actors": 0,
}

if stage_error is None:
    total_row = spark.sql(f"""
        SELECT
            COUNT(*)                                                    AS repository_days,
            COALESCE(SUM(issue_comment_events), 0)                      AS issue_comment_events,
            COALESCE(SUM(issue_conversation_comment_events), 0)         AS issue_conversation_comment_events,
            COALESCE(SUM(pr_conversation_comment_events), 0)            AS pr_conversation_comment_events,
            COALESCE(SUM(unclassified_conversation_comment_events), 0)  AS unclassified_conversation_comment_events,
            COALESCE(SUM(review_events), 0)                             AS review_events,
            COALESCE(SUM(review_state_known_events), 0)                 AS review_state_known_events,
            COALESCE(SUM(review_approved_events), 0)                    AS review_approved_events,
            COALESCE(SUM(review_changes_requested_events), 0)           AS review_changes_requested_events,
            COALESCE(SUM(review_commented_events), 0)                   AS review_commented_events,
            COALESCE(SUM(review_dismissed_events), 0)                   AS review_dismissed_events,
            COALESCE(SUM(review_other_state_events), 0)                 AS review_other_state_events,
            COALESCE(SUM(inline_review_comment_events), 0)              AS inline_review_comment_events,
            COALESCE(SUM(collaboration_events_total), 0)                AS collaboration_events_total,
            COALESCE(SUM(actor_known_events), 0)                        AS actor_known_events,
            COALESCE(SUM(unique_collaborating_actors), 0)               AS unique_collaborating_actors
        FROM {STAGING_TABLE}
    """).collect()[0]

    quality_metrics["repository_days"] = total_row["repository_days"]
    quality_metrics["issue_comment_events"] = total_row["issue_comment_events"]
    quality_metrics["issue_conversation_comment_events"] = total_row["issue_conversation_comment_events"]
    quality_metrics["pr_conversation_comment_events"] = total_row["pr_conversation_comment_events"]
    quality_metrics["unclassified_conversation_comment_events"] = total_row["unclassified_conversation_comment_events"]
    quality_metrics["review_events"] = total_row["review_events"]
    quality_metrics["review_state_known_events"] = total_row["review_state_known_events"]
    quality_metrics["review_approved_events"] = total_row["review_approved_events"]
    quality_metrics["review_changes_requested_events"] = total_row["review_changes_requested_events"]
    quality_metrics["review_commented_events"] = total_row["review_commented_events"]
    quality_metrics["review_dismissed_events"] = total_row["review_dismissed_events"]
    quality_metrics["review_other_state_events"] = total_row["review_other_state_events"]
    quality_metrics["inline_review_comment_events"] = total_row["inline_review_comment_events"]
    quality_metrics["collaboration_events_total"] = total_row["collaboration_events_total"]
    quality_metrics["actor_known_events"] = total_row["actor_known_events"]
    quality_metrics["unique_collaborating_actors"] = total_row["unique_collaborating_actors"]

print("Repository-days                    :", quality_metrics["repository_days"])
print("Issue comment events               :", quality_metrics["issue_comment_events"])
print("  on issues                        :", quality_metrics["issue_conversation_comment_events"])
print("  in pull request conversations    :", quality_metrics["pr_conversation_comment_events"])
print("  with an unknown target           :", quality_metrics["unclassified_conversation_comment_events"])
print("Review events                      :", quality_metrics["review_events"])
print("  with a known state               :", quality_metrics["review_state_known_events"])
print("  approved                         :", quality_metrics["review_approved_events"])
print("  changes requested                :", quality_metrics["review_changes_requested_events"])
print("  commented                        :", quality_metrics["review_commented_events"])
print("  dismissed                        :", quality_metrics["review_dismissed_events"])
print("  another state                    :", quality_metrics["review_other_state_events"])
print("Inline review comment events       :", quality_metrics["inline_review_comment_events"])
print("All collaboration events           :", quality_metrics["collaboration_events_total"])
print("Events with a known actor          :", quality_metrics["actor_known_events"])
print("Actors (person-repository-days)    :", quality_metrics["unique_collaborating_actors"])

## 10. Audit

One row is added to the Gold audit table for this run. The values are written in the order of the table's columns.

`source_rows_read` has one number per Silver table.

If the run failed, the audit row is written first. Then the notebook stops with an error, so that the Job task fails too.

In [0]:
# The Delta version of the Gold table after this run.
gold_table_version = spark.sql(
    f"DESCRIBE HISTORY {GOLD_TABLE} LIMIT 1"
).collect()[0]["version"]

if len(failed_checks) == 0:
    failed_checks_text = None
else:
    failed_checks_text = "; ".join(failed_checks)

# True when no gate failed. A run without rows has no failed gate.
checks_passed = stage_error is None and len(failed_checks) == 0

# The Silver rows this run used, per source table.
source_rows_read = {
    "issue_comment_events": issue_comment_row_count,
    "pull_request_review_events": review_row_count,
    "pull_request_review_comment_events": review_comment_row_count,
}

run_finished_at = datetime.now(timezone.utc)

audit_row = (
    str(uuid.uuid4()),                                  # audit_id
    run_id,                                             # run_id
    PIPELINE_NAME,                                      # pipeline_name
    GOLD_TABLE,                                         # gold_table
    processing_scope,                                   # processing_scope
    source_date_value,                                  # source_date
    affected_dates,                                     # affected_dates
    run_started_at,                                     # run_started_at
    run_finished_at,                                    # run_finished_at
    run_status,                                         # status
    error_message,                                      # error_message
    source_rows_read,                                   # source_rows_read
    {"missing_event_date": rows_without_event_date},    # rows_excluded
    published_rows,                                     # gold_rows_published
    gold_table_version,                                 # gold_table_version
    checks_passed,                                      # checks_passed
    failed_checks_text,                                 # failed_checks
    quality_metrics,                                    # quality_metrics
)

audit_schema = spark.table(AUDIT_TABLE).schema
audit_df = spark.createDataFrame([audit_row], schema=audit_schema)
audit_df.write.mode("append").saveAsTable(AUDIT_TABLE)

print("Audit row written:", AUDIT_TABLE)
print("run_id:", run_id)
print("status:", run_status)

if run_status == "NO_INPUT":
    print("WARNING: Silver has no rows in the scope of this run. The Gold table was not changed.")

if rows_without_event_date > 0:
    print("WARNING: rows without an event date that were left out:", rows_without_event_date)

if run_status == "FAILED":
    raise RuntimeError("repository_collaboration_metrics FAILED: " + error_message)

## 11. Results

These cells only read. They show the audit row of this run and the Gold table as it is after this run.

### 11.1 The audit row of this run

In [0]:
display(spark.sql(f"""
    SELECT *
    FROM {AUDIT_TABLE}
    WHERE run_id = '{run_id}'
      AND pipeline_name = '{PIPELINE_NAME}'
"""))

### 11.2 The Gold table per day

Expected on the real week, from the profiling run of 14 November 2026:

| Day | Rows | Collaboration events |
|---|---|---|
| 2024-02-01 | 75,896 | 449,194 |
| 2024-02-02 | 67,275 | 393,209 |
| 2024-02-03 | 37,985 | 171,153 |
| 2024-02-04 | 37,587 | 174,277 |
| 2024-02-05 | 82,181 | 438,816 |
| 2024-02-06 | 73,223 | 444,097 |
| 2024-02-07 | 70,573 | 426,772 |

In [0]:
display(spark.sql(f"""
    SELECT
        activity_date,
        COUNT(*)                                       AS repository_days,
        SUM(collaboration_events_total)                AS collaboration_events,
        SUM(issue_comment_events)                      AS issue_comment_events,
        SUM(issue_conversation_comment_events)         AS on_issues,
        SUM(pr_conversation_comment_events)            AS in_pr_conversations,
        SUM(unclassified_conversation_comment_events)  AS unknown_target,
        SUM(review_events)                             AS review_events,
        SUM(review_state_known_events)                 AS state_known,
        SUM(review_approved_events)                    AS approved,
        SUM(review_changes_requested_events)           AS changes_requested,
        SUM(review_commented_events)                   AS commented,
        SUM(review_dismissed_events)                   AS dismissed,
        SUM(review_other_state_events)                 AS other_state,
        SUM(inline_review_comment_events)              AS inline_review_comments,
        SUM(actor_known_events)                        AS actor_known_events,
        SUM(unique_collaborating_actors)               AS actors
    FROM {GOLD_TABLE}
    GROUP BY activity_date
    ORDER BY activity_date
"""))

### 11.3 The whole Gold table in one row

Expected after a run for all dates on the real week:

| Value | Expected |
|---|---|
| Rows | 444,720 |
| Collaboration events | 2,497,518 |
| Issue comment events | 1,357,324 |
| Of those, with an unknown target | 2 |
| Review events | 717,277 |
| Reviews with a known state | 717,277, because no review has a missing state |
| Inline review comment events | 422,917 |
| Events with a known actor | 2,497,518, so the coverage of the whole week is 100 |

The split by target and by state was counted before only on the raw events, before repeated events were removed. Silver has 3 issue comments and 8 reviews fewer than the raw events. So each value should be at most that much lower:

| Value | Raw events |
|---|---|
| Comments in pull request conversations | 838,179 |
| Comments on issues | 519,146 |
| Reviews: approved | 339,097 |
| Reviews: commented | 323,312 |
| Reviews: changes requested | 43,110 |
| Reviews: dismissed | 11,744 |
| Reviews: another state (`pending`) | 22 |

The number of actors was not measured before. It cannot be above the number of events with a known actor.

The coverage of the whole week is calculated from the two sums. It is not the average of the coverages of the rows.

In [0]:
display(spark.sql(f"""
    SELECT
        COUNT(*)                                       AS repository_days,
        COUNT(DISTINCT activity_date)                  AS days,
        SUM(collaboration_events_total)                AS collaboration_events,
        SUM(issue_comment_events)                      AS issue_comment_events,
        SUM(issue_conversation_comment_events)         AS on_issues,
        SUM(pr_conversation_comment_events)            AS in_pr_conversations,
        SUM(unclassified_conversation_comment_events)  AS unknown_target,
        SUM(review_events)                             AS review_events,
        SUM(review_state_known_events)                 AS state_known,
        SUM(review_approved_events)                    AS approved,
        SUM(review_changes_requested_events)           AS changes_requested,
        SUM(review_commented_events)                   AS commented,
        SUM(review_dismissed_events)                   AS dismissed,
        SUM(review_other_state_events)                 AS other_state,
        SUM(inline_review_comment_events)              AS inline_review_comments,
        SUM(actor_known_events)                        AS actor_known_events,
        SUM(unique_collaborating_actors)               AS actors,
        CASE
            WHEN SUM(collaboration_events_total) > 0
            THEN ROUND(100 * SUM(actor_known_events) / SUM(collaboration_events_total), 2)
        END                                            AS actor_coverage_pct_of_all_rows
    FROM {GOLD_TABLE}
"""))

### 11.4 A sample: the ten repository-days with the most collaboration events

A look at real rows. The busiest rows often belong to automated accounts: in this week 38% of the issue comment events come from logins that end in `[bot]`.

In [0]:
display(spark.sql(f"""
    SELECT *
    FROM {GOLD_TABLE}
    ORDER BY collaboration_events_total DESC, repo_id, activity_date
    LIMIT 10
"""))

## What this notebook wrote

- `gold.repository_collaboration_metrics`: the affected days were replaced, when the status is `SUCCEEDED`.
- `gold_staging.repository_collaboration_metrics`: the staging copy, overwritten on every run.
- `audit.gold_pipeline_runs`: one new row.

With a `test_schema`, all three are in that schema.

## How to read the table

- The numbers count events. An approving review event is not the same as "the pull request was approved".
- More comments do not mean better collaboration.
- Most issue comment events are pull request conversations, about 62% in this week.
- `unique_collaborating_actors` counts accounts, and an account can be a bot.
- 61% of the rows are built from one or two events. The count columns allow a reader to set a threshold.